# 06 - Evaluation: Models Evaluation


In this notebook, we perform an analysis of all trained models using:
- Metric distribution analysis (plots and summary statistics)

We do not perform comparison among models, we just evaluate each single model as it is.


## Import libraries and set the paths

In [1]:
from __future__ import annotations

import math

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from fraud_dynamic_ensemble.config import MODELS_DIR, FIGURES_DIR

2026-01-24 17:56:32.687 | INFO     | fraud_dynamic_ensemble.config:<module>:14 - PROJ_ROOT path is: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection


In [2]:
EXPERIMENT_NAME = "CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5"

In [3]:
models_results_path: Path = MODELS_DIR / EXPERIMENT_NAME
print(f"Loading results at path:\n\t{models_results_path}")

Loading results at path:
	/home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/models/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5


In [4]:
FIGURES_MODELS_COMPARISON_DIR = FIGURES_DIR / "EV_models_comparison_evaluation"
FIGURES_MODELS_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR = FIGURES_MODELS_COMPARISON_DIR / EXPERIMENT_NAME
FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_SINGLE_MODEL_EVALUATION_DIR = FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR / "single_models_evaluation"
FIGURES_SINGLE_MODEL_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
pd.set_option("display.max_columns", None)
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
sns.set_palette("tab10")

## Data Loading and Basic Overview

In [6]:
files = list(models_results_path.glob("*/resubstitution_metrics_summary.csv"))
resubstitution_df = pd.DataFrame()

print(f"Found {len(files)} files. Loading...")

if files:
    # Read and Concatenate
    # We use a generator expression inside concat for memory efficiency
    resubstitution_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    print("Success! Combined dataframe shape:", resubstitution_df.shape)
else:
    print(f"No files found in {models_results_path.absolute()}")

Found 23 files. Loading...
Success! Combined dataframe shape: (2300, 30)


In [7]:
resubstitution_df

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,fold_size,cv_tuning_mean_train_score,cv_tuning_std_train_score,cv_tuning_mean_val_score,cv_tuning_std_val_score,best_params,tuning_time,selected_features_indices,selected_features_names
0,CostSensitiveLearning___RandomizedSearchCV__ni...,1,1,KNOP,resubstitution,283,17676,31,58,0.995069,0.901274,0.829912,0.864122,0.998249,0.001751,0.914081,0.910197,0.862376,0.861615,0.966552,0.860887,22560,0.866747,0.005971,0.869068,0.022749,{'classifier__estimator__ccp_alpha': np.float6...,49.658297,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
1,CostSensitiveLearning___RandomizedSearchCV__ni...,1,2,KNOP,resubstitution,284,17686,21,57,0.995678,0.931148,0.832845,0.879257,0.998814,0.001186,0.915829,0.912062,0.878479,0.877064,0.965858,0.860910,22560,0.880571,0.007716,0.879193,0.030029,{'classifier__estimator__ccp_alpha': np.float6...,37.295071,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2,CostSensitiveLearning___RandomizedSearchCV__ni...,1,3,KNOP,resubstitution,286,17684,23,55,0.995678,0.925566,0.838710,0.880000,0.998701,0.001299,0.918705,0.915216,0.878910,0.877805,0.962738,0.856854,22560,0.874437,0.007910,0.877516,0.022190,{'classifier__estimator__ccp_alpha': np.float6...,47.781111,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
3,CostSensitiveLearning___RandomizedSearchCV__ni...,1,4,KNOP,resubstitution,281,17678,29,60,0.995069,0.906452,0.824047,0.863287,0.998362,0.001638,0.911205,0.907027,0.861796,0.860782,0.966037,0.863051,22560,0.867964,0.005753,0.868224,0.027638,{'classifier__estimator__ccp_alpha': np.float6...,49.188327,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
4,CostSensitiveLearning___RandomizedSearchCV__ni...,1,5,KNOP,resubstitution,285,17679,29,55,0.995346,0.907643,0.838235,0.871560,0.998362,0.001638,0.918299,0.914802,0.869907,0.869193,0.965912,0.849322,22560,0.868238,0.006911,0.867474,0.020704,{'classifier__estimator__ccp_alpha': np.float6...,50.497220,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2295,CostSensitiveLearning___RandomizedSearchCV__ni...,10,6,APriori,resubstitution,283,17675,33,57,0.995013,0.895570,0.832353,0.862805,0.998136,0.001864,0.915245,0.911483,0.860867,0.860269,0.963772,0.854117,22560,0.861620,0.003307,0.865230,0.016552,{'classifier__estimator__ccp_alpha': np.float6...,39.720943,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
2296,CostSensitiveLearning___RandomizedSearchCV__ni...,10,7,APriori,resubstitution,282,17691,17,58,0.995844,0.943144,0.829412,0.882629,0.999040,0.000960,0.914226,0.910283,0.882407,0.880523,0.967016,0.863151,22560,0.872562,0.007942,0.875066,0.024172,{'classifier__estimator__ccp_alpha': np.float6...,44.635312,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2297,CostSensitiveLearning___RandomizedSearchCV__ni...,10,8,APriori,resubstitution,293,17689,18,48,0.996343,0.942122,0.859238,0.898773,0.998983,0.001017,0.929110,0.926479,0.897901,0.896915,0.965748,0.872707,22561,0.890648,0.010944,0.892177,0.016732,{'classifier__estimator__ccp_alpha': np.float6...,45.145511,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2298,CostSensitiveLearning___RandomizedSearchCV__ni...,10,9,APriori,resubstitution,284,17683,24,57,0.995512,0.922078,0.832845,0.875193,0.998645,0.001355,0.915745,0.911985,0.874086,0.872914,0.969883,0.874898,22561,0.881255,0.009741,0.883311,0.031299,{'

In [8]:
files = list(models_results_path.glob("*/generalization_metrics_summary.csv"))
generalization_df = pd.DataFrame()

print(f"Found {len(files)} files. Loading...")

if files:
    # Read and Concatenate
    # We use a generator expression inside concat for memory efficiency
    generalization_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    print("Success! Combined dataframe shape:", generalization_df.shape)
else:
    print(f"No files found in {models_results_path.absolute()}")

Found 23 files. Loading...
Success! Combined dataframe shape: (2300, 25)


In [9]:
generalization_df

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,score_time,fold_size,selected_features_indices,selected_features_names
0,CostSensitiveLearning___RandomizedSearchCV__ni...,1,1,KNOP,generalization,44,2415,45,3,0.980854,0.494382,0.936170,0.647059,0.981707,0.018293,0.958939,0.958668,0.672788,0.638181,0.971969,0.947270,9.851874,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
1,CostSensitiveLearning___RandomizedSearchCV__ni...,1,2,KNOP,generalization,40,2424,36,7,0.982848,0.526316,0.851064,0.650407,0.985366,0.014634,0.918215,0.915756,0.661678,0.642115,0.942350,0.824992,6.699205,2507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2,CostSensitiveLearning___RandomizedSearchCV__ni...,1,3,KNOP,generalization,39,2435,25,8,0.986837,0.609375,0.829787,0.702703,0.989837,0.010163,0.909812,0.906286,0.704823,0.696133,0.969495,0.856795,10.615066,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
3,CostSensitiveLearning___RandomizedSearchCV__ni...,1,4,KNOP,generalization,39,2439,21,8,0.988432,0.650000,0.829787,0.728972,0.991463,0.008537,0.910625,0.907030,0.728785,0.723151,0.940369,0.835085,8.374084,2507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
4,CostSensitiveLearning___RandomizedSearchCV__ni...,1,5,KNOP,generalization,41,2379,80,7,0.965297,0.338843,0.854167,0.485207,0.967466,0.032534,0.910817,0.909053,0.525351,0.470695,0.925054,0.850371,13.076344,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2295,CostSensitiveLearning___RandomizedSearchCV__ni...,10,6,APriori,generalization,42,2379,80,6,0.965696,0.344262,0.875000,0.494118,0.967466,0.032534,0.921233,0.920072,0.536572,0.479823,0.961341,0.840800,7.278489,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
2296,CostSensitiveLearning___RandomizedSearchCV__ni...,10,7,APriori,generalization,39,2394,65,9,0.970483,0.375000,0.812500,0.513158,0.973566,0.026434,0.893033,0.889395,0.540214,0.500059,0.940215,0.726804,7.918153,2507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2297,CostSensitiveLearning___RandomizedSearchCV__ni...,10,8,APriori,generalization,40,2364,95,7,0.959298,0.296296,0.851064,0.439560,0.961366,0.038634,0.906215,0.904535,0.488182,0.423521,0.950283,0.804343,14.182767,2506,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2298,CostSensitiveLearning___RandomizedSearchCV__ni...,10,9,APriori,generalization,38,2420,39,9,0.980846,0.493506,0.808511,0.612903,0.984140,0.015860,0.896325,0.892013,0.623090,0.603672,0.912103,0.727781,10.022541,2506,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."


In [10]:
df_results = pd.concat([resubstitution_df, generalization_df], ignore_index=True)
df_results.head(10)

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,fold_size,cv_tuning_mean_train_score,cv_tuning_std_train_score,cv_tuning_mean_val_score,cv_tuning_std_val_score,best_params,tuning_time,selected_features_indices,selected_features_names,score_time
0,CostSensitiveLearning___RandomizedSearchCV__ni...,1,1,KNOP,resubstitution,283,17676,31,58,0.995069,0.901274,0.829912,0.864122,0.998249,0.001751,0.914081,0.910197,0.862376,0.861615,0.966552,0.860887,22560,0.866747,0.005971,0.869068,0.022749,{'classifier__estimator__ccp_alpha': np.float6...,49.658297,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",NaN
1,CostSensitiveLearning___RandomizedSearchCV__ni...,1,2,KNOP,resubstitution,284,17686,21,57,0.995678,0.931148,0.832845,0.879257,0.998814,0.001186,0.915829,0.912062,0.878479,0.877064,0.965858,0.860910,22560,0.880571,0.007716,0.879193,0.030029,{'classifier__estimator__ccp_alpha': np.float6...,37.295071,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
2,CostSensitiveLearning___RandomizedSearchCV__ni...,1,3,KNOP,resubstitution,286,17684,23,55,0.995678,0.925566,0.838710,0.880000,0.998701,0.001299,0.918705,0.915216,0.878910,0.877805,0.962738,0.856854,22560,0.874437,0.007910,0.877516,0.022190,{'classifier__estimator__ccp_alpha': np.float6...,47.781111,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",NaN
3,CostSensitiveLearning___RandomizedSearchCV__ni...,1,4,KNOP,resubstitution,281,17678,29,60,0.995069,0.906452,0.824047,0.863287,0.998362,0.001638,0.911205,0.907027,0.861796,0.860782,0.966037,0.863051,22560,0.867964,0.005753,0.868224,0.027638,{'classifier__estimator__ccp_alpha': np.float6...,49.188327,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
4,CostSensitiveLearning___RandomizedSearchCV__ni...,1,5,KNOP,resubstitution,285,17679,29,55,0.995346,0.907643,0.838235,0.871560,0.998362,0.001638,0.918299,0.914802,0.869907,0.869193,0.965912,0.849322,22560,0.868238,0.006911,0.867474,0.020704,{'classifier__estimator__ccp_alpha': np.float6...,50.497220,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",NaN
5,CostSensitiveLearning___RandomizedSearchCV__ni...,1,6,KNOP,resubstitution,290,17685,23,50,0.995955,0.926518,0.852941,0.888208,0.998701,0.001299,0.925821,0.922948,0.886939,0.886152,0.966098,0.872252,22560,0.892196,0.014738,0.893241,0.030683,{'classifier__estimator__ccp_alpha': np.float6...,33.314633,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
6,CostSensitiveLearning___RandomizedSearchCV__ni...,1,7,KNOP,resubstitution,277,17692,16,63,0.995623,0.945392,0.814706,0.875197,0.999096,0.000904,0.906901,0.902203,0.875486,0.872982,0.962991,0.851854,22560,0.879885,0.006623,0.878340,0.028940,{'classifier__estimator__ccp_alpha': np.float6...,41.007021,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
7,CostSensitiveLearning___RandomizedSearchCV__ni...,1,8,KNOP,resubstitution,291,17675,32,50,0.995457,0.900929,0.853372,0.876506,0.998193,0.001807,0.925783,0.922946,0.874527,0.874193,0.976720,0.873436,22561,0.885172,0.008375,0.882779,0.020070,{'classifier__estimator__ccp_alpha': np.float6...,48.944786,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",NaN
8,CostSensitiveLearning___RandomizedSearchCV__ni...,1,9,KNOP,resubstitution,282,17685,22,59,0.995512,0.927632,0.826979,0.874419,0.998758,0.001242,0.912869,0.908819,0.873633,0.872141,0.964971,0.847780,22561,0.873386,0.003477,0.872159,0.020792,{'classifier__estimator__ccp_alpha': np.float6...,48.737960,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,..."

In [11]:
df_results.tail(10)

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,fold_size,cv_tuning_mean_train_score,cv_tuning_std_train_score,cv_tuning_mean_val_score,cv_tuning_std_val_score,best_params,tuning_time,selected_features_indices,selected_features_names,score_time
4590,CostSensitiveLearning___RandomizedSearchCV__ni...,10,1,APriori,generalization,39,2387,73,8,0.967690,0.348214,0.829787,0.490566,0.970325,0.029675,0.900056,0.897309,0.525298,0.476746,0.933450,0.749671,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",15.718847
4591,CostSensitiveLearning___RandomizedSearchCV__ni...,10,2,APriori,generalization,41,2419,41,6,0.981252,0.500000,0.872340,0.635659,0.983333,0.016667,0.927837,0.926176,0.652472,0.626763,0.935859,0.722300,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",2.654266
4592,CostSensitiveLearning___RandomizedSearchCV__ni...,10,3,APriori,generalization,40,2396,64,7,0.971679,0.384615,0.851064,0.529801,0.973984,0.026016,0.912524,0.910452,0.561180,0.517337,0.919651,0.720878,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",1.295122
4593,CostSensitiveLearning___RandomizedSearchCV__ni...,10,4,APriori,generalization,40,2343,117,7,0.950538,0.254777,0.851064,0.392157,0.952439,0.047561,0.901751,0.900326,0.449801,0.374095,0.922946,0.789969,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",11.602670
4594,CostSensitiveLearning___RandomizedSearchCV__ni...,10,5,APriori,generalization,38,2371,88,10,0.960909,0.301587,0.791667,0.436782,0.964213,0.035787,0.877940,0.873691,0.474119,0.420719,0.930781,0.766161,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",15.142726
4595,CostSensitiveLearning___RandomizedSearchCV__ni...,10,6,APriori,generalization,42,2379,80,6,0.965696,0.344262,0.875000,0.494118,0.967466,0.032534,0.921233,0.920072,0.536572,0.479823,0.961341,0.840800,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8...",7.278489
4596,CostSensitiveLearning___RandomizedSearchCV__ni...,10,7,APriori,generalization,39,2394,65,9,0.970483,0.375000,0.812500,0.513158,0.973566,0.026434,0.893033,0.889395,0.540214,0.500059,0.940215,0.726804,2507,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",7.918153
4597,CostSensitiveLearning___RandomizedSearchCV__ni...,10,8,APriori,generalization,40,2364,95,7,0.959298,0.296296,0.851064,0.439560,0.961366,0.038634,0.906215,0.904535,0.488182,0.423521,0.950283,0.804343,2506,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",14.182767
4598,CostSensitiveLearning___RandomizedSearchCV__ni...,10,9,APriori,generalization,38,2420,39,9,0.980846,0.493506,0.808511,0.612903,0.984140,0.015860,0.896325,0.892013,0.623090,0.603672,0.912103,0.727781,2506,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",10.022541
4599,CostSensitiveLearning___RandomizedSearchCV__ni...,10,10,APriori,generalization,41,2382,77,6,0.966879,0.347458,0.872340,0.496970,0.968686,0.031314,0.920513,0.919252,0.538617,0.483104,0.930010,0.786274,2506,NaN,NaN,NaN,NaN,NaN,NaN,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',...",6.902661


## Fix the metrics and general settings

In [12]:
# Define your metrics
metrics_to_analyze = [
    "balanced_accuracy",
    "mcc",
    "average_precision",
    "f1",
]
print(f"Selected metrics:\n\t{metrics_to_analyze}")

# Get list of unique models
unique_models = df_results["model"].unique()
print(f"Selected models:\n\t{unique_models}")


Selected metrics:
	['balanced_accuracy', 'mcc', 'average_precision', 'f1']
Selected models:
	['KNOP' 'KNORAE' 'APosteriori' 'KNeighborsClassifier' 'Exponential'
 'LogitBoostClassifier' 'BalancedRandomForestClassifier'
 'DecisionTreeClassifier' 'StackingClassifier' 'DESKL' 'XGBClassifier'
 'Logarithmic' 'RUSBoostClassifier' 'MLA' 'RandomForestClassifier'
 'KNORAU' 'MLPClassifier' 'ExtraTreesClassifier' 'VotingClassifier' 'RRC'
 'METADES' 'DESP' 'APriori']


## Learning Stability (Train vs. Test Trajectory)

### Objective
To analyze the stability of the model's performance across the 10 distinct random initializations (Iterations). This visualization helps determine if the model's performance is consistent regardless of the random seed or if it fluctuates wildly.

### Methodology
We aggregate the performance over the 10 folds within each iteration to compute a robust estimate for that specific random seed.
* **X-Axis:** The Iteration Number (1 to 10).
* **Y-Axis:** The Metric Score.
* **Solid Lines:** The **Mean** score across the 10 folds for that iteration.
* **Shaded Areas:** The **Standard Deviation** ($\pm 1 \sigma$) across the 10 folds, representing the intra-iteration variance.

### Interpretation Guide
1.  **Parallel Trajectories:**
    * *Ideal Scenario:* The Train (Resubstitution) and Test (Generalization) lines move in parallel. This indicates consistent generalization behavior.
2.  **Wide Shaded Bands:**
    * *Warning Sign:* Large shaded areas indicate high variance between folds within a single iteration. The model is highly sensitive to the specific data partition (e.g., a "lucky" or "unlucky" fold).
3.  **Divergence:**
    * *Overfitting Sign:* If the Train line stays high and flat while the Test line dips significantly in specific iterations, those seeds produced non-generalizable models.

In [13]:
def plot_learning_curves(df, model_name, metrics_list, save_path):
    """
    Plot train vs. generalization trajectories across iterations for a given model and metrics.

    This function filters results for ``model_name`` and, for each metric in ``metrics_list``,
    aggregates scores by ``iteration`` and ``split`` (mean and standard deviation across folds).
    It then produces one figure per metric showing:
    - mean trajectory lines for ``"resubstitution"`` (train) and ``"generalization"`` (test),
    - shaded bands representing ±1 standard deviation across folds.

    One image is saved per metric under ``save_path`` and a dictionary of the aggregated
    statistics tables is returned.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing (at minimum) the columns: ``"model"``, ``"iteration"``,
        ``"split"``, and the metric columns referenced by ``metrics_list``.
    model_name : str
        Model identifier used to filter the input DataFrame (matched against ``df["model"]``).
    metrics_list : Sequence[str]
        Metric column names to visualize (e.g., ``["mcc", "roc_auc", "f1"]``). Metrics that
        are not present in ``df`` are skipped with a warning.
    save_path : pathlib.Path
        Output directory where figures are saved. The function assumes the directory exists
        and is writable.

    Returns
    -------
    Dict[str, pandas.DataFrame]
        Dictionary mapping each successfully processed metric name to the corresponding
        aggregated statistics DataFrame with columns:
        ``["iteration", "split", "mean", "std"]``.

        If no rows are found for ``model_name``, an empty dictionary is returned.

    Notes
    -----
    - Data are filtered as ``df[df["model"] == model_name]``.
    - Aggregation is computed via::

          df.groupby(["iteration", "split"])[metric].agg(["mean", "std"]).reset_index()

    - The plot assumes metric values are bounded in ``[0, 1]`` (fixed y-axis limits
      ``0`` to ``1.05``). Adjust if you include metrics outside this range (e.g., MCC in
      ``[-1, 1]``).
    - One image is saved per metric with filename::

          <model_name>_learning_curves_<metric>.png

    Examples
    --------
    >>> import pandas as pd
    >>> import numpy as np
    >>> from pathlib import Path
    >>> df = pd.DataFrame(
    ...     {
    ...         "model": ["A"] * 4,
    ...         "iteration": [1, 1, 2, 2],
    ...         "fold": [1, 2, 1, 2],
    ...         "split": ["resubstitution", "generalization", "resubstitution", "generalization"],
    ...         "roc_auc": [0.95, 0.90, 0.96, 0.91],
    ...     }
    ... )
    >>> stats = plot_learning_curves(df, "A", ["roc_auc"], Path("."))
    >>> "roc_auc" in stats
    True
    """

    # 1. Filter Data
    model_data = df[df["model"] == model_name].copy()
    if model_data.empty:
        print(f"Skipping {model_name}: No data found.")
        return {}

    # Store stats for return
    all_stats = {}

    # 2. Iterate over each metric
    for metric in metrics_list:

        # Check if metric exists in dataframe
        if metric not in model_data.columns:
            print(f"Warning: Metric '{metric}' not found in dataframe. Skipping.")
            continue

        # 3. Aggregation: Calculate Mean and Std per Iteration & Split
        # Collapse the 10 folds into summary stats
        stats_df = model_data.groupby(["iteration", "split"])[metric].agg(["mean", "std"]).reset_index()
        all_stats[metric] = stats_df

        # 4. Setup Plot
        plt.figure(figsize=(12, 6))
        sns.set_style("whitegrid")

        # Define colors for splits
        colors = {"resubstitution": "#d62728", "generalization": "#1f77b4"} # Red for Train, Blue for Test
        labels = {"resubstitution": "Train (Resubstitution)", "generalization": "Test (Generalization)"}

        # 5. Plot Lines and Shaded Areas
        for split in ["resubstitution", "generalization"]:
            subset = stats_df[stats_df["split"] == split]

            # Plot Mean Line
            plt.plot(
                subset["iteration"],
                subset["mean"],
                marker="o",
                label=labels[split],
                color=colors[split],
                linewidth=2
            )

            # Plot Standard Deviation Shade
            plt.fill_between(
                subset["iteration"],
                subset["mean"] - subset["std"],
                subset["mean"] + subset["std"],
                color=colors[split],
                alpha=0.15 # Light transparency
            )

        # 6. Formatting
        plt.ylim([0, 1.05])
        plt.title(f"Learning Stability Analysis: {model_name} ({metric})", fontsize=16, fontweight="bold", pad=15)
        plt.xlabel("Iteration", fontweight="bold")
        plt.ylabel(f"Average {metric} Score", fontweight="bold")
        plt.xticks(range(1, 11)) # Ensure integer ticks for iterations 1-10
        plt.yticks(ticks=np.arange(0, 1.1, 0.1))
        plt.legend(loc="best", frameon=True)
        plt.grid(True, linestyle="--", alpha=0.6)

        # 7. Save
        fig_filename = f"{model_name}_learning_curves_{metric}.png"
        plt.savefig(save_path / fig_filename, dpi=300, bbox_inches="tight")
        plt.close()

        print(f" -> Learning curves plot saved: {save_path / fig_filename}")

    return all_stats

In [14]:
for model in unique_models:
    model_name_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_name_path.mkdir(parents=True, exist_ok=True)

    plot_learning_curves(df_results, model, metrics_to_analyze, model_name_path)

 -> Learning curves plot saved: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_learning_curves_balanced_accuracy.png
 -> Learning curves plot saved: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_learning_curves_mcc.png
 -> Learning curves plot saved: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_learning_curves_average_precision.png
 -> Learning curves p

## Overfitting & Generalization Gap Assessment
### Objective
To rigorously assess the model's ability to generalize to unseen data by quantifying the performance drop between the training phase (Resubstitution) and the testing phase (Generalization).

### Methodology: The Generalization Gap
For each of the 100 data partitions (10 Iterations × 10 Folds), we calculate the **Generalization Gap** ($\delta$) for every metric:

$$\delta_{metric} = \text{Score}_{train} - \text{Score}_{test}$$

* **$\delta \approx 0$:** Indicates a robust model that learns generalizable patterns (Ideal).
* **$\delta > 0$:** Indicates **Overfitting**. The model is memorizing the training data and failing to perform as well on new data.
* **$\delta < 0$:** Indicates **Underfitting** or a non-representative easy test split.

### Visualization Strategy
We utilize a hybrid **Boxplot + Strip Plot** to visualize these gaps:
1.  **Boxplot:** Displays the statistical summary (Median, IQR) of the gap distribution, providing a quick view of the "average" overfitting severity.
2.  **Strip Plot:** Overlays the raw data points (100 folds) as jittered dots. This reveals the density and detects specific "outlier folds" where the model may have failed significantly.
3.  **Reference Line:** A red dashed line at $y=0$ serves as the baseline for perfect generalization.

### Interpretation Guide
* **Tight Cluster at 0:** The model is stable and trustworthy.
* **Large Positive Spread:** The model is highly sensitive to the specific data split and likely over-parameterized.
* **High Outliers:** Individual dots floating high above the boxplot indicate specific partitions where the model failed to generalize, suggesting potential data quality issues in those specific folds.

In [15]:
def analyze_generalization_gap(df, model_name, metrics_list, save_path):
    """
    Calculate and visualize the generalization gap (train minus test) for a given model.

    This helper quantifies potential overfitting by computing, for each metric in
    ``metrics_list``, the difference:

        gap = resubstitution - generalization

    using per-(iteration, fold) paired values. The function then visualizes the gap
    distribution across metrics using a boxplot (summary statistics) overlaid with a
    strip plot (per-split points). It also writes:
    - a PNG figure to ``save_path``, and
    - a CSV of summary statistics (mean, std, min, max) per metric.

    Parameters
    ----------
    df : pandas.DataFrame
        Input table containing at least the columns:
        ``"model"``, ``"iteration"``, ``"fold"``, and ``"split"``. For each metric in
        ``metrics_list``, the DataFrame must contain a corresponding numeric column.
        The ``"split"`` column is expected to include the labels ``"resubstitution"``
        and ``"generalization"`` to enable paired subtraction.
    model_name : str
        Model identifier used to filter ``df`` via ``df["model"] == model_name``.
        The same value is used to build output filenames.
    metrics_list : Sequence[str]
        List of metric column names to include in the analysis (e.g.,
        ``["f1", "average_precision", "roc_auc"]``).
    save_path : pathlib.Path
        Output directory where the plot and CSV will be saved. This function assumes
        the directory exists.

    Returns
    -------
    summary_stats : pandas.DataFrame
        Per-metric summary statistics of the computed gaps with columns
        ``["mean", "std", "min", "max"]`` and one row per metric. If no data are found
        for ``model_name``, the function returns ``None`` (early exit).

    Notes
    -----
    - The gap is computed only for (iteration, fold) pairs where both split values are
      available after pivoting. Missing split entries will propagate as missing values
      in the subtraction.
    - Interpretation: positive gaps indicate better training performance than test
      performance (a common symptom of overfitting for that metric).
    - The plot is saved as ``<model_name>_gap_analysis.png`` and the statistics as
      ``<model_name>_gap_statistics.csv`` under ``save_path``.

    Examples
    --------
    >>> import pandas as pd
    >>> from pathlib import Path
    >>> df = pd.DataFrame(
    ...     {
    ...         "model": ["M1", "M1", "M1", "M1"],
    ...         "iteration": [1, 1, 1, 1],
    ...         "fold": [1, 1, 2, 2],
    ...         "split": ["resubstitution", "generalization", "resubstitution", "generalization"],
    ...         "f1": [0.90, 0.80, 0.88, 0.86],
    ...         "roc_auc": [0.95, 0.92, 0.94, 0.93],
    ...     }
    ... )
    >>> out = analyze_generalization_gap(df, "M1", ["f1", "roc_auc"], Path("."))
    >>> (out.loc["f1", "mean"] >= 0.0) and (out.shape[1] == 4)
    True
    """

    # 1. Filter Data for specific model
    model_data = df[df["model"] == model_name].copy()

    if model_data.empty:
        print(f"Skipping {model_name}: No data found.")
        return

    # 2. Pivot Data to align Train/Test for calculating the difference
    # We create a table where we can subtract 'generalization' from 'resubstitution' directly
    pivot_df = model_data.pivot_table(
        index=["iteration", "fold"],
        columns="split",
        values=metrics_list
    )

    # 3. Calculate the Gap
    # Gap = Resubstitution (Train) - Generalization (Test)
    gap_data = pd.DataFrame(index=pivot_df.index)
    for metric in metrics_list:
        gap_data[metric] = pivot_df[metric]["resubstitution"] - pivot_df[metric]["generalization"]

    # 4. Melt for Plotting
    gap_long = gap_data.melt(var_name="Metric", value_name="Generalization Gap")

    # 5. Plotting
    plt.figure(figsize=(14, 8))
    sns.set_style("whitegrid")

    # A. Boxplot for Statistics (White box, black lines)
    ax = sns.boxplot(
        data=gap_long,
        x="Metric",
        y="Generalization Gap",
        color="white",
        linecolor="#333333",
        width=0.6,
        fliersize=0,        # Hide outliers here (we show them in the strip plot)
        linewidth=1.5
    )

    # B. Strip Plot for Density & Outliers (Blue dots)
    sns.stripplot(
        data=gap_long,
        x="Metric",
        y="Generalization Gap",
        color="#1f77b4",
        alpha=0.4,          # Transparency allows seeing overlapping points
        jitter=0.25,        # Spreads dots horizontally
        size=4,
        ax=ax
    )

    # C. Reference Line (Zero Gap)
    plt.axhline(0, color="#d62728", linestyle="--", linewidth=2, alpha=0.8, label="Ideal Generalization (Gap=0)")

    # Styling
    plt.title(f"Generalization Gap Analysis: {model_name}\n(Positive Values = Overfitting)",
              fontsize=16, fontweight="bold", pad=20)
    plt.ylabel("Performance Drop (Train Score - Test Score)", fontweight="bold")
    plt.xlabel("Metric", fontweight="bold")
    plt.grid(True, axis="y", alpha=0.5, linestyle="--")
    plt.legend(loc="upper right", frameon=True)

    # 6. Save Plot
    fig_filename = f"{model_name}_gap_analysis.png"
    plt.savefig(save_path / fig_filename, dpi=300, bbox_inches="tight")
    plt.close()

    # 7. Save Summary Statistics CSV
    # Transpose describe() so metrics are rows, easier to read
    summary_stats = gap_data.describe().T[["mean", "std", "min", "max"]]
    summary_csv_name = f"{model_name}_gap_statistics.csv"
    summary_stats.to_csv(save_path / summary_csv_name)

    print(f"Gap Analysis for {model_name} completed.")
    print(f" -> Plot saved to: {save_path / fig_filename}")
    print(f" -> Stats saved to: {save_path / summary_csv_name}\n")

    return summary_stats

In [16]:
for model in unique_models:
    model_name_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_name_path.mkdir(parents=True, exist_ok=True)

    analyze_generalization_gap(df_results, model, metrics_to_analyze, model_name_path)

Gap Analysis for KNOP completed.
 -> Plot saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_gap_analysis.png
 -> Stats saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_gap_statistics.csv

Gap Analysis for KNORAE completed.
 -> Plot saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNORAE/KNORAE_gap_analysis.png
 -> Stats saved to: /home/leonard

## ROC Space Stability (The "Cloud" Check)

### Objective
To assess the reliability and consistency of the model by visualizing its "operating point" stability on unseen data (Generalization split). Unlike scalar metrics (e.g., Accuracy), this analysis reveals how the model balances Sensitivity vs. Specificity across different data partitions.

### Methodology
For each of the 100 data partitions (10 Iterations $\times$ 10 Folds), we map the model's performance into the 2D ROC Space:
* **X-Axis:** False Positive Rate ($\text{FPR} = \frac{\text{FP}}{\text{FP} + \text{TN}}$) or $1 - \text{Specificity}$.
* **Y-Axis:** True Positive Rate ($\text{TPR} = \frac{\text{TP}}{\text{TP} + \text{FN}}$) or $\text{Recall}$.

### Visualization Strategy
1.  **Scatter Cloud:** We plot 100 distinct points (blue dots). A tight cloud indicates a stable model, while a dispersed cloud indicates high variance.
2.  **Centroid ($\star$):** A large red star represents the average operating point of the model across all folds.
3.  **Reference Lines:** * **Diagonal (Grey Dashed):** Represents a random classifier ($\text{AUC} = 0.5$).
    * **Ideal Point (Green Cross):** The top-left corner $(0, 1)$ representing perfect prediction.

### Interpretation Guide
By observing the shape and position of the "Cloud", we can diagnose specific stability issues:

* **Tight Cluster ("Bullet Hole"):**
    * *Verdict:* **Reliable.**
    * *Meaning:* The model is extremely stable. It makes the same trade-offs regardless of how the data is split.

* **Diagonal Streak:**
    * *Verdict:* **Unstable Thresholding.**
    * *Meaning:* The model is sensitive to class balance differences in specific folds, trading off Precision for Recall unpredictably.

* **Wide Dispersion ("Shotgun Blast"):**
    * *Verdict:* **High Variance.**
    * *Meaning:* The model is highly sensitive to noise. It works well on some data splits (top-left) but fails on others (bottom-right). This often suggests the need for better regularization or feature selection.

* **Points near Diagonal:**
    * *Verdict:* **Random Guessing.**
    * *Meaning:* On these specific folds, the model failed to learn any useful patterns.

In [17]:
def analyze_roc_stability(df, model_name, save_path):
    """
    Analyze ROC-space stability of a single model on the generalization split.

    This helper filters the input results table to a single ``model_name`` and the
    ``"generalization"`` split, then computes per-row ROC-space coordinates:

    - True Positive Rate (TPR): ``tp / (tp + fn)``
    - False Positive Rate (FPR): ``fp / (fp + tn)``

    A small epsilon is added to denominators to reduce the risk of division-by-zero
    in edge cases. The function visualizes the resulting (FPR, TPR) point cloud as a
    scatter plot, overlays the centroid (mean FPR/TPR) as a highlighted marker, and
    includes standard ROC-space reference elements (random-guess diagonal and the
    ideal point (0, 1)). It also exports a CSV with summary statistics, including the
    Euclidean distance from the centroid to the ideal point.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing evaluation results. Must include:
        ``"model"``, ``"split"``, ``"tp"``, ``"fp"``, ``"tn"``, and ``"fn"`` columns.
        The ``"split"`` column must contain the value ``"generalization"`` for the
        desired evaluation subset.
    model_name : str
        Model identifier used to filter rows via ``df["model"] == model_name``.
        Also used to build output filenames.
    save_path : pathlib.Path
        Directory where outputs are written. The function assumes the directory
        exists and is writable.

    Returns
    -------
    stats : pandas.DataFrame
        Single-row DataFrame summarizing ROC-space stability with columns:
        ``["Model", "Mean_TPR", "Std_TPR", "Mean_FPR", "Std_FPR", "Distance_to_Ideal"]``.
        If no matching generalization rows are found, the function returns ``None``.

    Notes
    -----
    - This analysis uses only the ``"generalization"`` split to reflect expected
      deployment behavior.
    - TPR and FPR are computed from confusion-matrix counts. If a fold contains no
      positives or no negatives, denominators can be zero; an epsilon (``1e-9``) is
      added to mitigate division-by-zero.
    - The centroid is computed as the mean of per-row TPR and FPR values.
    - Outputs:
      - Figure: ``<model_name>_roc_stability_cloud.png``
      - Statistics: ``<model_name>_roc_stability_stats.csv``

    Examples
    --------
    >>> import pandas as pd
    >>> from pathlib import Path
    >>> df = pd.DataFrame(
    ...     {
    ...         "model": ["M1", "M1"],
    ...         "split": ["generalization", "generalization"],
    ...         "tp": [80, 75],
    ...         "fn": [20, 25],
    ...         "fp": [10, 12],
    ...         "tn": [890, 888],
    ...     }
    ... )
    >>> out = analyze_roc_stability(df, "M1", Path("."))
    >>> (out is None) or (0.0 <= float(out["Mean_TPR"].iloc[0]) <= 1.0)
    True
    """

    # 1. Filter Data: Generalization Split Only
    # We focus on the test set to evaluate true reliability
    model_data = df[(df["model"] == model_name) & (df["split"] == "generalization")].copy()

    if model_data.empty:
        print(f"Skipping {model_name}: No generalization data found.")
        return

    # 2. Calculate ROC Coordinates (TPR vs FPR) per fold
    # We add a tiny epsilon (1e-9) to the denominator to prevent DivisionByZero errors
    # in the rare edge case where a fold has 0 positives or 0 negatives.
    model_data["tpr_calc"] = model_data["tp"] / (model_data["tp"] + model_data["fn"] + 1e-9)
    model_data["fpr_calc"] = model_data["fp"] / (model_data["fp"] + model_data["tn"] + 1e-9)

    # 3. Calculate Centroid (Mean Point)
    mean_tpr = model_data["tpr_calc"].mean()
    mean_fpr = model_data["fpr_calc"].mean()

    # 4. Create Plot
    plt.figure(figsize=(10, 10)) # Square aspect ratio is standard for ROC
    sns.set_style("whitegrid")

    # Plot the "Cloud" of individual folds
    plt.scatter(
        model_data["fpr_calc"],
        model_data["tpr_calc"],
        c="#1f77b4",
        alpha=0.6,    # Transparency helps visualize density where points overlap
        s=60,
        edgecolor="white",
        label="Individual Folds (100)"
    )

    # Plot the Centroid
    plt.scatter(
        mean_fpr,
        mean_tpr,
        c="#d62728", # Red
        s=300,
        marker="*",
        edgecolor="black",
        zorder=10,
        label=f"Mean Point (TPR={mean_tpr:.2f}, FPR={mean_fpr:.2f})"
    )

    # Plot Reference Lines
    plt.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=2, label="Random Guess")
    plt.scatter(0, 1, c="green", s=100, marker="P", label="Ideal Point (0,1)")

    # Styling
    plt.title(f"Stability Analysis: {model_name}\n(ROC Space Distribution)", fontsize=16, fontweight="bold")
    plt.xlabel("False Positive Rate (1 - Specificity)", fontweight="bold")
    plt.ylabel("True Positive Rate (Sensitivity)", fontweight="bold")
    plt.xlim(-0.02, 1.02)
    plt.ylim(-0.02, 1.02)
    plt.grid(True, which="both", linestyle="--", linewidth=0.5)

    # Legend Positioning
    plt.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.1),
        ncol=2,
        frameon=True,
        fontsize=12,
        shadow=True
    )
    plt.subplots_adjust(bottom=0.2) # Ensure space for the legend

    # 5. Save Figure
    fig_filename = f"{model_name}_roc_stability_cloud.png"
    plt.savefig(save_path / fig_filename, dpi=300, bbox_inches="tight")
    plt.close()

    # 6. Save Statistics
    # We calculate the Euclidean distance to the ideal point (0,1) as a summary metric
    stats = pd.DataFrame({
        "Model": [model_name],
        "Mean_TPR": [mean_tpr],
        "Std_TPR": [model_data["tpr_calc"].std()],
        "Mean_FPR": [mean_fpr],
        "Std_FPR": [model_data["fpr_calc"].std()],
        "Distance_to_Ideal": [np.sqrt(mean_fpr**2 + (1-mean_tpr)**2)]
    })

    csv_filename = f"{model_name}_roc_stability_stats.csv"
    stats.to_csv(save_path / csv_filename, index=False)

    print(f"ROC Stability Analysis for {model_name} completed.")
    print(f" -> Plot saved to: {save_path / fig_filename}")
    print(f" -> Stats saved to: {save_path / csv_filename}\n")

    return stats

In [18]:
for model in unique_models:
    model_name_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_name_path.mkdir(parents=True, exist_ok=True)

    analyze_roc_stability(df_results, model, model_name_path)

ROC Stability Analysis for KNOP completed.
 -> Plot saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_roc_stability_cloud.png
 -> Stats saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP/KNOP_roc_stability_stats.csv

ROC Stability Analysis for KNORAE completed.
 -> Plot saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNORAE/KNORAE_roc_stability_clou

## Partition Stability (Heatmap Grid)

### Objective
To diagnose whether the model's performance is consistent across all data partitions or if it relies on specific "lucky" random splits. This analysis visualizes the score variability across every single Iteration and Fold.

### Methodology
We construct a **Heatmap Grid** for each metric:
* **Grid Structure:** 10 Rows (Iterations) $\times$ 10 Columns (Folds).
* **Color Intensity:** Represents the metric score (fixed range $0.0$ to $1.0$) using the **"brg"** palette.
    * **Green:** High Performance ($\approx 1.0$).
    * **Red:** Medium Performance ($\approx 0.5$).
    * **Blue:** Low Performance ($\approx 0.0$).

### Interpretation Guide
By scanning the grid patterns, we can detect three types of behavior:

1.  **Uniform Green:**
    * *Verdict:* **Robust & High Performing.**
    * *Meaning:* The model consistently achieves high scores regardless of how the data is sliced.

2.  **"TV Static" (Random Variation):**
    * *Verdict:* **Normal Variance.**
    * *Meaning:* Slight color variations (shades of green or light green) are expected, provided there are no deep red or blue patches.

3.  **Blue/Red Stripes (Rows or Columns):**
    * *Verdict:* **Suspicious Data Sensitivity.**
    * *Meaning:*
        * **Row Stripe:** A specific Iteration (Random Seed) created a partition where the Test set was consistently harder.
        * **Column Stripe:** A specific Fold number is problematic across iterations.

4.  **Single Blue Cell:**
    * *Verdict:* **"Black Swan" Event.**
    * *Meaning:* A specific combination of data broke the model (e.g., score dropped from 0.9 to 0.4). This warrants investigation into that specific fold's data distribution.

In [19]:
def analyze_partition_stability(df, model_name, metrics_list, save_path):
    """
    Visualize partition stability across iterations and folds using a grid of heatmaps.

    This function analyzes performance variability for a single ``model_name`` on the
    ``"generalization"`` split by creating one heatmap per metric in ``metrics_list``.
    For each metric, it pivots the filtered data into an (iteration × fold) matrix and
    renders a heatmap using a fixed color scale in the range ``[0.0, 1.0]`` with the
    ``"nrg"`` colormap. A shared horizontal colorbar is added at the bottom of the
    figure to ensure consistent interpretation across metrics.

    If ``metrics_list`` contains multiple metrics, subplots are arranged in a compact
    grid with two columns (and enough rows to fit all metrics). Any unused subplot axes
    are removed.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing evaluation results. Must include columns:
        ``"model"``, ``"split"``, ``"iteration"``, and ``"fold"``, plus all metric
        columns specified in ``metrics_list``. The function filters rows where
        ``df["model"] == model_name`` and ``df["split"] == "generalization"``.
    model_name : str
        Model identifier used to filter rows via ``df["model"] == model_name``.
        Also used to build the output filename.
    metrics_list : Sequence[str]
        List of metric column names to visualize. Each entry must correspond to a
        numeric column in ``df``. Heatmaps are created in the same order as provided.
    save_path : pathlib.Path
        Directory where the heatmap grid image is saved. The function assumes the
        directory exists and is writable.

    Returns
    -------
    None
        This function returns ``None``. If no matching rows are found for the selected
        model and split, it prints a message and returns early.

    Notes
    -----
    - The visualization uses a fixed intensity range (``vmin=0.0``, ``vmax=1.0``) for
      all heatmaps and the shared colorbar. Ensure the metrics plotted are naturally
      bounded in ``[0, 1]`` (e.g., AUC, F1, accuracy) for meaningful interpretation.
    - Heatmaps are annotated with values formatted to two decimal places.
    - Output file:
      - ``<model_name>_stability_heatmap_grid.png`` saved under ``save_path``.

    Examples
    --------
    >>> import pandas as pd
    >>> from pathlib import Path
    >>> df = pd.DataFrame(
    ...     {
    ...         "model": ["M1", "M1", "M1", "M1"],
    ...         "split": ["generalization"] * 4,
    ...         "iteration": [1, 1, 2, 2],
    ...         "fold": [1, 2, 1, 2],
    ...         "roc_auc": [0.91, 0.89, 0.92, 0.90],
    ...         "f1": [0.70, 0.68, 0.72, 0.69],
    ...     }
    ... )
    >>> analyze_partition_stability(df, "M1", ["roc_auc", "f1"], Path("."))
    >>> True
    True
    """

    # 1. Filter Data: Model + Generalization Split
    mask = (df["model"] == model_name) & (df["split"] == "generalization")
    model_data = df[mask].copy()

    if model_data.empty:
        print(f"Skipping {model_name}: No generalization data found.")
        return

    # 2. Setup Subplots Grid
    num_metrics = len(metrics_list)
    cols = 2 if num_metrics > 1 else 1
    rows = math.ceil(num_metrics / cols)

    # Dynamic figure size
    fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))

    # Flatten axes
    if num_metrics > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    # Global Title
    fig.suptitle(f"Partition Stability Analysis: {model_name}",
                 fontsize=20, fontweight='bold', y=0.98)

    # 3. Loop through metrics
    for i, metric in enumerate(metrics_list):
        ax = axes[i]

        # Prepare Pivot Table
        heatmap_data = model_data.pivot(index="iteration", columns="fold", values=metric)

        # Draw Heatmap
        sns.heatmap(
            heatmap_data,
            ax=ax,
            annot=True,
            fmt=".2f",
            cmap="brg",
            cbar=False,
            linewidths=.5,
            vmin=0.0,
            vmax=1.0
        )

        ax.set_title(f"Metric: {metric.upper()}", fontsize=14, fontweight="bold")
        ax.set_xlabel("Fold", fontsize=14)
        ax.set_ylabel("Iteration", fontsize=14)

    # 4. Hide empty subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # 5. Add Common Horizontal Colorbar at Bottom
    plt.tight_layout()
    fig.subplots_adjust(bottom=0.12)

    # Define position: [left, bottom, width, height]
    cbar_ax = fig.add_axes([0.15, 0.06, 0.7, 0.025])

    norm = plt.Normalize(vmin=0.0, vmax=1.0)
    sm = plt.cm.ScalarMappable(cmap="brg", norm=norm)
    sm.set_array([])

    # Define ticks every 0.05
    ticks_range = np.arange(0, 1.05, 0.05)

    cbar = fig.colorbar(sm, cax=cbar_ax, orientation="horizontal", ticks=ticks_range)
    cbar.set_label("Score Intensity (0.0 - 1.0) | Blue=Low, Green=High", fontsize=12, fontweight="bold")

    # 6. Save Figure
    fig_filename = f"{model_name}_stability_heatmap_grid.png"
    plt.savefig(save_path / fig_filename, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Heatmap Grid for {model_name} -> Saved at path:\n\t {save_path}\n")

In [20]:
for model in unique_models:
    model_name_path = FIGURES_SINGLE_MODEL_EVALUATION_DIR / model
    model_name_path.mkdir(parents=True, exist_ok=True)

    analyze_partition_stability(df_results, model, metrics_to_analyze, model_name_path)

Heatmap Grid for KNOP -> Saved at path:
	 /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNOP

Heatmap Grid for KNORAE -> Saved at path:
	 /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/KNORAE

Heatmap Grid for APosteriori -> Saved at path:
	 /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/single_models_evaluation/APosteriori

Heatmap Grid for KNeighborsClassifier -> Saved at path:
	 /home/leonardosaccotelli/De